# Feature Engineering & Validation

## 1. Goal
Generate operational proxy targets, engineer time/location features, and validate for data leakage.

## 2. Imports

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to sys.path
project_root = Path(os.getcwd()).parent.parent
sys.path.append(str(project_root))

from notebooks.config import *
from backend.app.ai.features.target_engineering import (
    generate_response_duration_target, 
    generate_congestion_proxy, 
    generate_deployment_target
)


## 3. Load Cleaned Data

In [2]:
intermediate_path = PROCESSED_DATA_DIR / "intermediate_cleaned.parquet"
df = pd.read_parquet(intermediate_path)
print(f"Loaded {len(df)} rows.")

Loaded 8173 rows.


## 4. Target Generation
Using operational proxies for realistic predictions.

In [3]:
# 1. Response Duration
df[RESPONSE_TIME_TARGET] = generate_response_duration_target(df)

# 2. Congestion Proxy
df[CONGESTION_TARGET] = generate_congestion_proxy(df)

# 3. Deployment Load Classification
df[DEPLOYMENT_TARGET] = generate_deployment_target(df)

display(df[[RESPONSE_TIME_TARGET, CONGESTION_TARGET, DEPLOYMENT_TARGET]].head())

,resolution_time_minutes,congestion_proxy_score,deployment_load_class
0,NaN,3.00000,LOW
1,10.377589,3.17296,MEDIUM
2,NaN,1.00000,LOW
3,NaN,8.00000,MEDIUM
4,NaN,1.00000,LOW


## 5. Feature Engineering
Extracting temporal and spatial features.

In [4]:
# Time features from start_datetime
# Ensure it's datetime
df['start_datetime'] = pd.to_datetime(df['start_datetime'], utc=True)

df['hour_of_day'] = df['start_datetime'].dt.hour
df['day_of_week'] = df['start_datetime'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['is_peak_hour'] = df['hour_of_day'].isin([8,9,10, 17,18,19]).astype(int)

# Operational Features
# Severity encoded
priority_map = {'unknown': 0, 'low': 1, 'medium': 2, 'high': 3, 'critical': 4}
df['priority_encoded'] = df['priority'].str.lower().map(priority_map).fillna(0)

# Closure
df['is_closed'] = df['requires_road_closure'].astype(str).str.lower() == 'true'
df['is_closed'] = df['is_closed'].astype(int)


## 6. Leakage Validation
Ensure no future information (e.g. resolved_datetime) leaks into predictive features.

In [5]:
# List of features meant for training (excluding targets and leakage columns)
leakage_cols = ['resolved_datetime', 'closed_datetime', 'modified_datetime', 'resolved_by_id']
# Validate none of these are going into our primary feature list
primary_features = ['latitude', 'longitude', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_peak_hour', 'priority_encoded', 'is_closed']

for col in primary_features:
    assert col not in leakage_cols, f"Leakage detected: {col} is a future event column!"
    
print("Leakage validation passed! Primary features are safe.")

Leakage validation passed! Primary features are safe.


## 7. Export Processed Data & Feature Snapshot

In [6]:
# Final export
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
df.to_parquet(CLEANED_DATA_PATH, index=False)
print(f"Saved {len(df)} engineered rows to {CLEANED_DATA_PATH}")

# Feature Snapshot Export
snapshot = df[primary_features + [RESPONSE_TIME_TARGET, CONGESTION_TARGET, DEPLOYMENT_TARGET]].dtypes.astype(str).to_dict()
snapshot_path = PROCESSED_DATA_DIR / "feature_snapshot.json"
import json
with open(snapshot_path, "w") as f:
    json.dump(snapshot, f, indent=4)
print(f"Saved feature snapshot to {snapshot_path}")

Saved 8173 engineered rows to C:\Users\akash\gridwise-ai\data\processed\cleaned_dataset.parquet
Saved feature snapshot to C:\Users\akash\gridwise-ai\data\processed\feature_snapshot.json
